In [3]:
import pandas as pd
df=pd.read_csv(r'C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\Preprocessed_Data.csv')

In [4]:
import joblib
import os

In [5]:
df.columns

Index(['Unnamed: 0.5', 'Unnamed: 0.4', 'Unnamed: 0.3', 'Unnamed: 0.2',
       'Unnamed: 0.1', 'Unnamed: 0', 'inningNumber', 'oversActual',
       'pitchLine', 'pitchLength', 'isFour', 'isSix', 'isWicket', 'byes',
       'legbyes', 'wides', 'noballs', 'run', 'totalRuns', 'totalWickets',
       'shotType', 'time_of_day', 'Ground Name', 'Batsman_Name',
       'Batsman_Role', 'Batsman_Batting_Style', 'Batsman_Playing_Role',
       'Bowler_Name', 'Bowler_Role', 'Bowler_Bowling_Style',
       'Bowler_Playing_Role', 'isBoundary', 'isGoodBall', 'isDeathOver',
       'line_x_length'],
      dtype='object')

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
import joblib
import warnings
warnings.filterwarnings('ignore')


In [7]:
def load_and_clean(df):
    bool_cols = ['isFour', 'isSix', 'isWicket']
    for c in bool_cols:
        if c in df.columns:
            df[c] = df[c].astype(int)
    df = df.apply(pd.to_numeric, errors='ignore')
    return df

df=load_and_clean(df)
df.head()


,Unnamed: 0.5,Unnamed: 0.4,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,inningNumber,oversActual,pitchLine,pitchLength,...,Batsman_Batting_Style,Batsman_Playing_Role,Bowler_Name,Bowler_Role,Bowler_Bowling_Style,Bowler_Playing_Role,isBoundary,isGoodBall,isDeathOver,line_x_length
0,0,0,0,0,0,0,1,0.1,1,2,...,1,6,47,1,4,3,0,1,0,2
1,1,1,1,1,1,1,1,0.2,1,4,...,1,6,47,1,4,3,0,1,0,4
2,2,2,2,2,2,2,1,0.3,1,2,...,1,6,47,1,4,3,0,1,0,2
3,3,3,3,3,3,3,1,0.4,1,2,...,1,6,47,1,4,3,0,1,0,2
4,4,4,4,4,4,4,1,0.5,1,2,...,1,6,47,1,4,3,0,1,0,2


In [8]:
def add_targets(df, goodball_run_threshold=1):
    df = df.copy()
    df['isBoundary'] = ((df.get('isFour', 0) == 1) | (df.get('isSix', 0) == 1)).astype(int)
    df['isGoodBall'] = ((df['run'] <= goodball_run_threshold) & (df['isBoundary'] == 0)).astype(int)
    return df

In [9]:
df = add_targets(df, goodball_run_threshold=1)
print("Counts -> boundaries:", df['isBoundary'].sum(), "good balls:", df['isGoodBall'].sum())

Counts -> boundaries: 3627 good balls: 21285


In [10]:
features = ['inningNumber','oversActual','pitchLine','pitchLength','shotType','time_of_day','Batsman_Batting_Style','Bowler_Bowling_Style']

In [11]:
if 'oversActual' in df.columns:
    df['isDeathOver'] = (df['oversActual'] >= 16).astype(int)
    features.append('isDeathOver')

In [12]:
if 'pitchLine' in df.columns and 'pitchLength' in df.columns:
        df['line_x_length'] = df['pitchLine'] * df['pitchLength']
        features.append('line_x_length')

In [13]:
features

['inningNumber',
 'oversActual',
 'pitchLine',
 'pitchLength',
 'shotType',
 'time_of_day',
 'Batsman_Batting_Style',
 'Bowler_Bowling_Style',
 'isDeathOver',
 'line_x_length']

In [14]:
X = df[features].fillna(0).astype(float)

In [15]:
y=df['isBoundary'].values

In [16]:
X.shape

(27273, 10)

In [17]:
# X.to_csv(r"C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\model_feature_engineered_datasets/df_model_boundary.csv", index=False)

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

In [19]:
classes = np.unique(y_train)
print(len(classes))
cw = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, cw))
print("Class weights:", class_weights)

2
Class weights: {np.int64(0): np.float64(0.5767075491647282), np.int64(1): np.float64(3.759131633356306)}


In [20]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight=class_weights)
rf.fit(X_train_sc, y_train)
y_pred_prob_rf = rf.predict_proba(X_test_sc)[:,1]
y_pred_rf = (y_pred_prob_rf > 0.5).astype(int)
print("RF report:\n", classification_report(y_test, y_pred_rf))
print("Confusion:\n", confusion_matrix(y_test, y_pred_rf))
print("ROC AUC:", roc_auc_score(y_test, y_pred_prob_rf))


RF report:
               precision    recall  f1-score   support

           0       0.87      0.96      0.92      4730
           1       0.25      0.08      0.12       725

    accuracy                           0.85      5455
   macro avg       0.56      0.52      0.52      5455
weighted avg       0.79      0.85      0.81      5455

Confusion:
 [[4557  173]
 [ 668   57]]
ROC AUC: 0.6941462418896259


Class imbalance issue:

Only 725 out of 5455 are class 1 → ~13%

RF struggles to predict minority class (boundary / scoring ball)

ROC AUC = 0.694 → okay, shows model can distinguish somewhat but not great.

Precision & recall for class 1:

Precision = 0.25 → only 1 in 4 predicted boundaries is correct

Recall = 0.08 → most actual boundaries are missed

In [21]:
joblib.dump(rf, r"C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\models\predict_boundary_model.joblib")

['C:\\Users\\yaswa\\OneDrive\\Desktop\\artificial intelligence project\\models\\predict_boundary_model.joblib']

In [22]:
df.columns

Index(['Unnamed: 0.5', 'Unnamed: 0.4', 'Unnamed: 0.3', 'Unnamed: 0.2',
       'Unnamed: 0.1', 'Unnamed: 0', 'inningNumber', 'oversActual',
       'pitchLine', 'pitchLength', 'isFour', 'isSix', 'isWicket', 'byes',
       'legbyes', 'wides', 'noballs', 'run', 'totalRuns', 'totalWickets',
       'shotType', 'time_of_day', 'Ground Name', 'Batsman_Name',
       'Batsman_Role', 'Batsman_Batting_Style', 'Batsman_Playing_Role',
       'Bowler_Name', 'Bowler_Role', 'Bowler_Bowling_Style',
       'Bowler_Playing_Role', 'isBoundary', 'isGoodBall', 'isDeathOver',
       'line_x_length'],
      dtype='object')

In [23]:
y=df['isGoodBall']

In [24]:
# X.to_csv(r"C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\model_feature_engineered_datasets/df_model_good_ball.csv", index=False)

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [26]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)

In [27]:
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:,1]

In [28]:
import pandas as pd
import plotly.express as px

# Assuming you trained model_goodball using X_train
importances = rf.feature_importances_
feature_names = X_train.columns

# Combine into DataFrame
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False).head(15)

# Plot using Plotly
fig = px.bar(
    importance_df,
    x="Feature",
    y="Importance",
    title="Feature Importance – Random Forest (Good Ball Model)",
    text="Importance",
    color="Importance",
    color_continuous_scale="Blues"
)

fig.update_layout(
    title_font_size=20,
    xaxis_title_font_size=16,
    yaxis_title_font_size=16,
    plot_bgcolor="white",
    yaxis=dict(showgrid=True, gridcolor="lightgray"),
    xaxis_tickangle=-30
)

fig.show()


In [29]:
print("Random Forest Report:")
print(classification_report(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("ROC AUC:", roc_auc_score(y_test, y_prob_rf))

Random Forest Report:
              precision    recall  f1-score   support

           0       0.33      0.16      0.22      1198
           1       0.79      0.91      0.85      4257

    accuracy                           0.74      5455
   macro avg       0.56      0.53      0.53      5455
weighted avg       0.69      0.74      0.71      5455

Confusion Matrix:
 [[ 193 1005]
 [ 399 3858]]
ROC AUC: 0.6624573568899383


The Random Forest model is moderately accurate for detecting good balls, which is sufficient for recommendation purposes, but predicting poor/scoring balls is inherently difficult due to cricket’s stochastic nature and dataset imbalance. The model’s probabilistic outputs are more meaningful than strict classification for real-life use

In [30]:
joblib.dump(rf, r"C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\models\predict_good_ball_model_random_forest.joblib")

['C:\\Users\\yaswa\\OneDrive\\Desktop\\artificial intelligence project\\models\\predict_good_ball_model_random_forest.joblib']

In [31]:
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


In [32]:
input_dim = X_train_sc.shape[1]

In [33]:
ann_model = Sequential([
    Dense(64, activation='relu', input_dim=input_dim),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')  
])

In [34]:
ann_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [35]:
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


In [36]:
ann_model.fit(X_train_sc, y_train,
              validation_split=0.2,
              epochs=50,
              batch_size=64,
              callbacks=[es],
              verbose=1)

Epoch 1/50
273/273 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7782 - loss: 0.5463 - val_accuracy: 0.7793 - val_loss: 0.5312
Epoch 2/50
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7807 - loss: 0.5353 - val_accuracy: 0.7793 - val_loss: 0.5309
Epoch 3/50
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7807 - loss: 0.5331 - val_accuracy: 0.7793 - val_loss: 0.5306
Epoch 4/50
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7807 - loss: 0.5317 - val_accuracy: 0.7793 - val_loss: 0.5298
Epoch 5/50
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7807 - loss: 0.5307 - val_accuracy: 0.7793 - val_loss: 0.5286
Epoch 6/50
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7807 - loss: 0.5296 - val_accuracy: 0.7793 - val_loss: 0.5296
Epoch 7/50
273/273 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7807 - loss: 0.5281 - val_accuracy: 0.7793 - val_loss: 0.5295
Epoch 8/50
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7807 - loss: 0.5285 - val_accuracy: 0.

In [37]:
y_prob_ann = ann_model.predict(X_test_sc).flatten()
y_pred_ann = (y_prob_ann > 0.5).astype(int)

171/171 ━━━━━━━━━━━━━━━━━━━━ 0s 805us/step


In [38]:
print("ANN Report:")
print(classification_report(y_test, y_pred_ann))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_ann))
print("ROC AUC:", roc_auc_score(y_test, y_prob_ann))

ANN Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      1198
           1       0.78      1.00      0.88      4257

    accuracy                           0.78      5455
   macro avg       0.39      0.50      0.44      5455
weighted avg       0.61      0.78      0.68      5455

Confusion Matrix:
 [[   0 1198]
 [   0 4257]]
ROC AUC: 0.49376495866770354


Batsman_analysis_engine

In [39]:
df1=pd.read_csv(r'C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\batsman_profiles.csv')

In [40]:
df1.columns

Index(['Full Name', 'Run_Scored', 'Balls_Faced', 'Strike_Rate', 'Dismissals',
       'Dismissal_Rate', 'Boundary_percentage', 'Average'],
      dtype='object')

In [41]:
df.columns

Index(['Unnamed: 0.5', 'Unnamed: 0.4', 'Unnamed: 0.3', 'Unnamed: 0.2',
       'Unnamed: 0.1', 'Unnamed: 0', 'inningNumber', 'oversActual',
       'pitchLine', 'pitchLength', 'isFour', 'isSix', 'isWicket', 'byes',
       'legbyes', 'wides', 'noballs', 'run', 'totalRuns', 'totalWickets',
       'shotType', 'time_of_day', 'Ground Name', 'Batsman_Name',
       'Batsman_Role', 'Batsman_Batting_Style', 'Batsman_Playing_Role',
       'Bowler_Name', 'Bowler_Role', 'Bowler_Bowling_Style',
       'Bowler_Playing_Role', 'isBoundary', 'isGoodBall', 'isDeathOver',
       'line_x_length'],
      dtype='object')

In [42]:
df.to_csv(r'C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\Preprocessed_Data.csv')